# 🧪 InfoDev - Playground de Testes e Debug

Bem-vindo ao ambiente de desenvolvimento do **InfoDev**, um sistema multi-agente RAG projetado para extrair e raciocinar sobre dados de engenharia de software (issues, commits e e-mails).

Este notebook tem como objetivo testar os componentes do sistema de forma isolada (Módulos, Agentes e Bancos Vetoriais) antes de integrá-los no grafo de execução final (LangGraph).

### ⚙️ Configurações e Ambiente
* **Python Mínimo:** 3.11 (Garante compatibilidade com wheels C++ de IA)
* **Gerenciamento de Pacotes:** Certifique-se de que o seu ambiente virtual está ativo e instale as dependências com: `pip install -r requirements.txt`.
* **Variáveis de Ambiente:** O sistema depende do arquivo `.env` na raiz do projeto contendo as chaves `GROQ_API_KEY` e `MONGO_URI`.

In [2]:
# MÁGICA DO JUPYTER: Força o notebook a recarregar arquivos .py 
# automaticamente toda vez que você rodar uma célula.
%load_ext autoreload
%autoreload 2

import os
from dotenv import load_dotenv

# Garante que as credenciais do .env estão ativas no ambiente
load_dotenv(override=True)
print("Variáveis de ambiente carregadas.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Variáveis de ambiente carregadas.


## 🗄️ VectorStoreManager

Classe que gerencia os embeddings

## Area de Testes

Conexão com o banco e contagem de documentos

In [4]:
from pymongo import MongoClient

print("📊 --- Raio-X do Banco de Dados MongoDB ---")

# Conecta ao banco de dados (ajuste a URI se necessário)
MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "clean_shark"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

colecoes = ["rich_issues", "rich_commits", "rich_emails"]
projeto_alvo = "tez" # O projeto que você vai vetorizar

for col_name in colecoes:
    colecao = db[col_name]
    
    # Conta o total absoluto na coleção
    total_docs = colecao.count_documents({})
    
    # Conta apenas os documentos do projeto alvo
    total_projeto = colecao.count_documents({"project": projeto_alvo})
    
    # Busca um documento de exemplo para checar os campos
    exemplo = colecao.find_one()
    campos = list(exemplo.keys()) if exemplo else "Nenhum documento"
    
    # Tenta descobrir quais projetos únicos existem nessa coleção (pode demorar um pouquinho se o banco for gigante)
    try:
        projetos_unicos = colecao.distinct("project")
    except:
        projetos_unicos = ["Erro ao buscar"]
        
    print(f"\n📁 Coleção: {col_name}")
    print(f"  - Total de documentos: {total_docs}")
    print(f"  - Documentos do projeto '{projeto_alvo}': {total_projeto}")
    print(f"  - Projetos únicos encontrados: {projetos_unicos}")
    print(f"  - Campos disponíveis: {campos}")

print("\n✅ Diagnóstico concluído!")

📊 --- Raio-X do Banco de Dados MongoDB ---

📁 Coleção: rich_issues
  - Total de documentos: 22342
  - Documentos do projeto 'tez': 4133
  - Projetos únicos encontrados: ['nifi', 'pdfbox', 'phoenix', 'ranger', 'tez']
  - Campos disponíveis: ['_id', 'project', 'type', 'original_id', 'title', 'status', 'text_for_embedding', 'created_at']

📁 Coleção: rich_commits
  - Total de documentos: 33780
  - Documentos do projeto 'tez': 3659
  - Projetos únicos encontrados: ['nifi', 'pdfbox', 'phoenix', 'ranger', 'tez']
  - Campos disponíveis: ['_id', 'project', 'type', 'hash', 'date', 'text_for_embedding', 'files_touched']

📁 Coleção: rich_emails
  - Total de documentos: 225463
  - Documentos do projeto 'tez': 8942
  - Projetos únicos encontrados: ['nifi', 'pdfbox', 'phoenix', 'ranger', 'tez']
  - Campos disponíveis: ['_id', 'project', 'type', 'subject', 'text_for_embedding', 'date']

✅ Diagnóstico concluído!


Ingestão

In [3]:
import sys
sys.path.append('./src')

from Config import Config
from VectorStoreManager import VectorStoreManager

print("--- Inicializando as 3 Bases de Dados Vetoriais ---")

projeto_alvo = "tez"
filtro = {"project": projeto_alvo}
db_path = f"./vectorstores/{projeto_alvo}_db"

# Modelos escolhidos
MODELO_CODIGO = "jinaai/jina-embeddings-v2-base-code"
MODELO_TEXTO = "nomic-ai/nomic-embed-text-v1.5"

# 1. Instancia passando o modelo correto para cada domínio
manager_commits = VectorStoreManager(
    persist_directory=db_path, 
    collection_name="commits", 
    model_name=MODELO_CODIGO     # <--- Jina para código
)

manager_issues = VectorStoreManager(
    persist_directory=db_path, 
    collection_name="issues", 
    model_name=MODELO_TEXTO      # <--- Nomic para texto
)

manager_emails = VectorStoreManager(
    persist_directory=db_path, 
    collection_name="emails", 
    model_name=MODELO_TEXTO      # <--- Nomic para texto
)

DO_INGESTION = True

if DO_INGESTION:
    print("\n🚀 --- INICIANDO PIPELINE DE INGESTÃO ---")
    
    # Processa Commits (Vai baixar/usar o Jina)
    manager_commits.ingest_documents(
        mongo_collection_name=Config.COLLECTION_COMMITS, 
        doc_type="commit", 
        mongo_filter=filtro,
        batch_size=32
    )
    
    # Processa Issues (Vai baixar/usar o Nomic)
    manager_issues.ingest_documents(
        mongo_collection_name=Config.COLLECTION_ISSUES, 
        doc_type="issue", 
        mongo_filter=filtro,
        batch_size=32
    )
    
    # Processa E-mails (Usa o Nomic que já está na memória)
    manager_emails.ingest_documents(
        mongo_collection_name=Config.COLLECTION_EMAILS, 
        doc_type="email", 
        mongo_filter=filtro,
        batch_size=32
    )
    print("PIPELINE TOTAL CONCLUÍDO!")

--- Inicializando as 3 Bases de Dados Vetoriais ---


c:\projects\infodev\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\projects\infodev\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\mathe\.cache\huggingface\modules\transformers_modules\jinaai\jina-bert-v2-qk-post-norm\3baf9e3ac750e76e8edd3019170176884695fb94\configuration_bert.py:29: UserWarning: optimum is not installed. To use OnnxConfig and BertOnnxConfig, make sure that `optimum` package is installed
  warnings.warn("optimum is not installed. To use OnnxConfig and BertOnnxConfig, make sure that `optimum` package is installed")
c:\projects\infodev\./src\VectorStoreManager.py:3


🚀 --- INICIANDO PIPELINE DE INGESTÃO ---

Conectando ao MongoDB (clean_shark -> rich_commits)...
Buscando documentos...
Dividindo 1000 documentos...
Total gerado: 29321 chunks.
Iniciando vetorização (Lotes de 32)...


Vetorizando 'commits': 100%|██████████| 917/917 [2:57:09<00:00, 11.59s/it]  


Ingestão na coleção 'commits' concluída com sucesso!

Conectando ao MongoDB (clean_shark -> rich_issues)...
Buscando documentos...
Dividindo 1000 documentos...
Total gerado: 4860 chunks.
Iniciando vetorização (Lotes de 32)...


Vetorizando 'issues': 100%|██████████| 152/152 [27:20<00:00, 10.80s/it]


Ingestão na coleção 'issues' concluída com sucesso!

Conectando ao MongoDB (clean_shark -> rich_emails)...
Buscando documentos...
Dividindo 1000 documentos...
Total gerado: 5422 chunks.
Iniciando vetorização (Lotes de 32)...


Vetorizando 'emails': 100%|██████████| 170/170 [50:35<00:00, 17.86s/it]


Ingestão na coleção 'emails' concluída com sucesso!
PIPELINE TOTAL CONCLUÍDO!


In [ ]:
import sys
sys.path.append('./src') # Aponta o notebook para a pasta src

from Config import Config
from VectorStoreManager import VectorStoreManager

print("--- Inicializando o Gerenciador de Banco Vetorial ---")

project = "tez"

print(f"Projeto selecionado: {project}")

# 1. Instancia o manager apontando para a pasta local do ChromaDB e nomeando a coleção
vsm = VectorStoreManager(
    persist_directory="./vectorstores/" + project + "_db", 
    collection_name=project + "_collection"
)

# 3. Teste de Busca (Retriever)
print("\n--- Testando a Busca Semântica Local ---")
retriever = vsm.get_retriever(k=3)

# Testando uma query
query_teste = " "

print(f"🔍 Buscando por: '{query_teste}'\n")
documentos_encontrados = retriever.invoke(query_teste)

# Exibe os resultados limpos
for i, doc in enumerate(documentos_encontrados, 1):
    print(f"📄 DOCUMENTO {i} | Origem: {doc.metadata.get('source')}")
    print(f"{doc.page_content[:300]}...\n")

--- Inicializando o Gerenciador de Banco Vetorial ---
Projeto selecionado: tez


c:\projects\infodev\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 